# Testeo de la API mediante Python

Proyecto: **Inventario de equipos computacionales**

Vamos a probar las dos APIs del proyecto desde Python usando la librería `requests`:

| API | Recurso | Protección |
|---|---|---|
| Pública | `/api/processors`, `/api/rams` | ninguna |
| Privada | `/api/devices`, `/api/me`, `/api/stats` | middleware `auth:sanctum` (token) |

Esto sirve para dos cosas: **demostrar que la API funciona** (y que el middleware
realmente bloquea), y dejar lista la base para hacer **carga masiva y automatizada
de datos** más adelante.

> **Antes de ejecutar:** deja corriendo el servidor en otra terminal con
> `php artisan serve`. Si no está corriendo, todas las celdas fallarán con
> `ConnectionError`.

## Instalar la librería Requests

`requests` nos permite hacer peticiones `HTTP` hacia la API.

In [53]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


d:\xampp\htdocs\xampp\proyectobackend\.venv\Scripts\python.exe: No module named pip


## Configuración

Aquí van la URL base y las credenciales del usuario administrador
(las mismas que define `database/seeders/UserSeeder.php`).

In [54]:
import requests

API = "http://127.0.0.1:8000/api"

EMAIL = "admin@ucsc.cl"
PASSWORD = "12345678"

print("Apuntando a:", API)

Apuntando a: http://127.0.0.1:8000/api


## 1. API pública: leer sin token

`/api/processors` y `/api/rams` no tienen middleware, así que responden
sin necesidad de autenticarse. Fíjate que no se manda ningún header `Authorization`.

In [55]:
# GET /api/processors -> lista paginada, ruta PÚBLICA
procesadores = requests.get(f"{API}/processors", timeout=15)
procesadores.raise_for_status()

print("Status:", procesadores.status_code)
for p in procesadores.json()["data"]:
    print(f'  [{p["id"]}] {p["brand"]} {p["model"]}')

# GET /api/rams -> también pública
rams = requests.get(f"{API}/rams", timeout=15)
rams.raise_for_status()

print("\nRAMs:")
for r in rams.json()["data"]:
    print(f'  [{r["id"]}] {r["type"]} {r["capacity"]}')

Status: 200
  [1] Intel Core i5-10300h
  [2] Amd Ryzen 7445hs

RAMs:
  [1] DDR5 16GB
  [2] DDR4 8GB


## 2. La API privada rechaza el acceso sin token

Antes de iniciar sesión, probamos que el middleware `auth:sanctum` está haciendo
su trabajo: `/api/devices` debe responder **401 Unauthorized**.

In [56]:
sin_token = requests.get(
    f"{API}/devices",
    headers={"Accept": "application/json"},
    timeout=15,
)

print("Status:", sin_token.status_code)
print("Respuesta:", sin_token.json())

assert sin_token.status_code == 401, "La ruta privada debería rechazar sin token"
print("\nCorrecto: el middleware auth:sanctum bloqueó la petición.")

Status: 401
Respuesta: {'message': 'No autenticado. Debes enviar un token válido en el header Authorization: Bearer {token}.'}

Correcto: el middleware auth:sanctum bloqueó la petición.


## 3. Inicio de sesión y obtención del token

`POST /api/login` nos devuelve un token personal de Sanctum. Ese token hay que
mandarlo en el header `Authorization: Bearer {token}` en cada petición privada.

Usamos un `requests.Session()` para no tener que repetir el header en cada llamada.

In [57]:
respuesta = requests.post(
    f"{API}/login",
    json={
        "email": EMAIL,
        "password": PASSWORD,
        "device_name": "notebook-python",
    },
    timeout=15,
)
respuesta.raise_for_status()

datos = respuesta.json()
token = datos["token"]

print("Usuario:", datos["user"]["name"], "| rol:", datos["user"]["rol"])
print("Token:", token[:25] + "...")

# Sesión con el token ya puesto en los headers
sesion = requests.Session()
sesion.headers.update({
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
})

Usuario: Administrador | rol: admin
Token: 8|ri4HyYZGcmHGVaKj42m1pwX...


## 4. API privada: ahora sí entramos

Con el token puesto, las mismas rutas que antes daban 401 responden 200.

In [58]:
# GET /api/me -> el usuario dueño del token
yo = sesion.get(f"{API}/me", timeout=15)
yo.raise_for_status()
print("Soy:", yo.json()["name"], "-", yo.json()["email"])

# GET /api/devices -> equipos (requiere token)
equipos = sesion.get(f"{API}/devices", timeout=15)
equipos.raise_for_status()

print("\nEquipos encontrados:", equipos.json()["total"])
for e in equipos.json()["data"]:
    marca = e["brand"]["name"] if e.get("brand") else "?"
    depto = e["department"]["name"] if e.get("department") else "?"
    print(f'  [{e["id"]}] {e["name"]} | marca: {marca} | depto: {depto}')

Soy: Administrador - admin@ucsc.cl

Equipos encontrados: 2
  [6] aaaa | marca: Lenovo | depto: Computación
  [7] prueba1 | marca: HP | depto: Informática


## 5. Middleware de rol: `/api/stats` es solo para admin

`/api/stats` lleva dos middleware encadenados: `auth:sanctum` **y** `role:admin`.
Como iniciamos sesión con el administrador, responde 200.

Si inicias sesión con un supervisor, esta misma celda devolvería **403**.

In [59]:
stats = sesion.get(f"{API}/stats", timeout=15)

print("Status:", stats.status_code)

if stats.status_code == 200:
    print("Resumen del inventario:", stats.json())
elif stats.status_code == 403:
    print("403 - este usuario no es admin:", stats.json())

Status: 200
Resumen del inventario: {'total_equipos': 2, 'equipos_por_depto': {'1': 1, '3': 1}, 'equipos_por_procesador': {'1': 1, '2': 1}}


## 6. Insertar un procesador (POST)

Ahora el CRUD completo sobre la API pública. Primero creamos un registro.

In [60]:
procesador = {
    "brand": "Intel",
    "model": "Core i7-13700K (creado desde Python)",
}

# POST: crea el registro
respuesta = requests.post(f"{API}/processors", json=procesador, timeout=15)
respuesta.raise_for_status()

creado = respuesta.json()
procesador_id = creado["id"]

print("Status:", respuesta.status_code)
print("Creado:", creado)

Status: 201
Creado: {'brand': 'Intel', 'model': 'Core i7-13700K (creado desde Python)', 'updated_at': '2026-11-10T18:15:17.000000Z', 'created_at': '2026-11-10T18:15:17.000000Z', 'id': 8}


## 7. Editar el procesador (PUT)

In [61]:
# PUT: actualiza el mismo registro.
# Se envían todos los campos porque la validación los exige.
procesador["brand"] = "AMD"
procesador["model"] = "Ryzen 9 7950X (actualizado desde Python)"

respuesta = requests.put(
    f"{API}/processors/{procesador_id}",
    json=procesador,
    timeout=15,
)
respuesta.raise_for_status()

print("Status:", respuesta.status_code)
print("Actualizado:", respuesta.json())

Status: 200
Actualizado: {'id': 8, 'brand': 'AMD', 'model': 'Ryzen 9 7950X (actualizado desde Python)', 'created_at': '2026-11-10T18:15:17.000000Z', 'updated_at': '2026-11-10T18:15:17.000000Z'}


## 8. Eliminar el procesador (DELETE)

In [62]:
# DELETE: elimina ese registro, responde 204 sin cuerpo
respuesta = requests.delete(f"{API}/processors/{procesador_id}", timeout=15)
respuesta.raise_for_status()

print("Status:", respuesta.status_code)
print(f"Eliminado el procesador con ID {procesador_id}")

Status: 204
Eliminado el procesador con ID 8


## 9. Validación: la API rechaza datos incompletos

Si mandamos un procesador sin `model`, Laravel responde **422** con los errores.

In [63]:
invalido = requests.post(
    f"{API}/processors",
    json={"brand": "Intel"},          # falta "model"
    headers={"Accept": "application/json"},
    timeout=15,
)

print("Status:", invalido.status_code)
print("Errores:", invalido.json()["errors"])

assert invalido.status_code == 422, "Debería rechazar los datos incompletos"
print("\nCorrecto: la validación funciona.")

Status: 422
Errores: {'model': ['The model field is required.']}

Correcto: la validación funciona.


## 10. Cierre de sesión

`POST /api/logout` revoca **solo** el token con el que se hizo la petición.

In [64]:
cierre = sesion.post(f"{API}/logout", timeout=15)

print("Logout:", cierre.status_code, cierre.json())

Logout: 200 {'message': 'Sesión cerrada, token revocado.'}


## 11. Comprobar que el token quedó inválido

Reutilizamos el mismo token después del logout: debe responder **401**.
Esto demuestra que los tokens se revocan de verdad y no quedan dando vueltas.

In [65]:
despues = requests.get(
    f"{API}/devices",
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    },
    timeout=15,
)

print("Status:", despues.status_code)
print("Respuesta:", despues.json())

assert despues.status_code == 401, "El token todavía es aceptado"
print("\nCorrecto: el token original está invalidado.")

Status: 401
Respuesta: {'message': 'No autenticado. Debes enviar un token válido en el header Authorization: Bearer {token}.'}

Correcto: el token original está invalidado.


## Resumen de lo que se comprobó

| # | Prueba | Resultado esperado |
|---|---|---|
| 1 | GET público sin token | 200 |
| 2 | GET privado sin token | 401 |
| 3 | POST /api/login | 200 + token |
| 4 | GET privado con token | 200 |
| 5 | GET /api/stats con admin | 200 (403 si no es admin) |
| 6 | POST /api/processors | 201 |
| 7 | PUT /api/processors/{id} | 200 |
| 8 | DELETE /api/processors/{id} | 204 |
| 9 | POST con datos incompletos | 422 |
| 10 | POST /api/logout | 200 |
| 11 | Reusar el token revocado | 401 |